## Spotify 음원 길이 가져오기

In [ ]:
import os
import pandas as pd
import requests
import base64
import time

# Spotify API 인증 정보
CLIENT_ID = "client_id"
CLIENT_SECRET = "client_secret"

# Spotify Access Token 가져오기
def get_token(cid, secret):
    resp = requests.post(
        "https://accounts.spotify.com/api/token",
        headers={"Authorization": "Basic " + base64.b64encode(f"{cid}:{secret}".encode()).decode()},
        data={"grant_type": "client_credentials"}
    )
    resp.raise_for_status()
    return resp.json()["access_token"]

# Spotify 요청 (429 처리 포함)
def spotify_get(url, headers, params=None):
    while True:
        resp = requests.get(url, headers=headers, params=params)
        if resp.status_code == 429:
            wait = int(resp.headers.get("Retry-After", 5))
            print(f"Rate limited — wait {wait}s")
            time.sleep(wait)
            continue
        if resp.status_code >= 400:
            return None
        return resp.json()

# 음원 길이 가져오기
def get_durations_batch(ids, token, market="US"):
    headers = {"Authorization": f"Bearer {token}"}
    durations = []
    url = "https://api.spotify.com/v1/tracks"
    for i in range(0, len(ids), 50):
        batch = ids[i:i+50]
        data = spotify_get(url, headers, params={"ids": ",".join(batch), "market": market})
        if data is None:
            return None
        for t in data.get("tracks", []):
            durations.append(int(t["duration_ms"] / 1000) if t and t.get("duration_ms") else None)
        time.sleep(0.2)
    return durations

# 폴더 경로 설정
input_dir = "global_top_chart_csv_2023-2025"
output_dir = "global_duration"
os.makedirs(output_dir, exist_ok=True)

# 토큰 발급
try:
    token = get_token(CLIENT_ID, CLIENT_SECRET)
except Exception as e:
    print(f"❌ Spotify 토큰 발급 실패: {e}")
    token = None

# 파일 처리
if token:
    for filename in sorted(os.listdir(input_dir)):
        if not filename.endswith(".csv"):
            continue
        filepath = os.path.join(input_dir, filename)
        try:
            df = pd.read_csv(filepath)
            df["track_id"] = df["uri"].str.split(":").str[-1]
            ids = df["track_id"].dropna().tolist()
            durations = get_durations_batch(ids, token)
            if durations is None or len(durations) != len(ids):
                print(f"❌ {filename} - 음원 길이 가져오기 실패")
                continue
            df["duration_sec"] = durations
            df.to_csv(os.path.join(output_dir, f"with_duration_{filename}"), index=False)
            print(f"✅ {filename} 저장 완료")
        except Exception as e:
            print(f"❌ {filename} - 처리 중 오류 발생: {e}")


✅ regional-global-weekly-2023-01-05.csv 저장 완료
✅ regional-global-weekly-2023-01-12.csv 저장 완료
✅ regional-global-weekly-2023-01-19.csv 저장 완료
✅ regional-global-weekly-2023-01-26.csv 저장 완료
✅ regional-global-weekly-2023-02-02.csv 저장 완료
✅ regional-global-weekly-2023-02-09.csv 저장 완료
✅ regional-global-weekly-2023-02-16.csv 저장 완료
✅ regional-global-weekly-2023-02-23.csv 저장 완료
✅ regional-global-weekly-2023-03-02.csv 저장 완료
✅ regional-global-weekly-2023-03-09.csv 저장 완료
✅ regional-global-weekly-2023-03-16.csv 저장 완료
✅ regional-global-weekly-2023-03-23.csv 저장 완료
✅ regional-global-weekly-2023-03-30.csv 저장 완료
✅ regional-global-weekly-2023-04-06.csv 저장 완료
✅ regional-global-weekly-2023-04-13.csv 저장 완료
✅ regional-global-weekly-2023-04-20.csv 저장 완료
✅ regional-global-weekly-2023-04-27.csv 저장 완료
✅ regional-global-weekly-2023-05-04.csv 저장 완료
✅ regional-global-weekly-2023-05-11.csv 저장 완료
✅ regional-global-weekly-2023-05-18.csv 저장 완료
✅ regional-global-weekly-2023-05-25.csv 저장 완료
✅ regional-global-weekly-2023-06-0

## Youtube Music 음원 길이 가져오기

In [ ]:
pip install isodate

In [ ]:
import pandas as pd
import requests
from urllib.parse import urlparse, parse_qs
import isodate

# ====== STEP 1: 설정 ======
API_KEY = "api_key"  # ← 본인의 API 키
CSV_FILE = "data/kr_top_combined.csv"  # ← 분석할 CSV 파일 경로
OUTPUT_FILE = "data/kr_top_with_duration.csv"

# ====== STEP 2: 유튜브 URL → Video ID 추출 ======
def extract_video_id(url):
    parsed_url = urlparse(url)
    if "youtube.com" in parsed_url.netloc:
        query = parse_qs(parsed_url.query)
        return query.get("v", [None])[0]
    elif "youtu.be" in parsed_url.netloc:
        return parsed_url.path.lstrip('/')
    return None

# ====== STEP 3: API 통해 영상 길이 가져오기 (ISO 8601 형식 → 초로 변환) ======
def get_video_duration(video_id):
    if not video_id:
        return None
    url = f"https://www.googleapis.com/youtube/v3/videos?part=contentDetails&id={video_id}&key={API_KEY}"
    response = requests.get(url)
    if response.status_code == 200:
        items = response.json().get("items")
        if items:
            iso_duration = items[0]["contentDetails"]["duration"]
            duration = isodate.parse_duration(iso_duration).total_seconds()
            return int(duration)
    return None

# ====== STEP 4: 실행 ======
df = pd.read_csv(CSV_FILE)

# Video ID 추출
df["Video ID"] = df["YouTube URL"].apply(extract_video_id)

# 영상 길이 가져오기 (약간 느릴 수 있음)
df["Duration (sec)"] = df["Video ID"].apply(get_video_duration)

# 저장
df.to_csv(OUTPUT_FILE, index=False)

print(f"🎉 완료! 영상 길이가 포함된 파일이 저장됨: {OUTPUT_FILE}")
